---

# Phase 10 · Section 4 — Feature Additions

### Objective

Add three features that improve usability and demonstrate additional frontend engineering: a client-side search bar, pagination on the projects page, and a dark mode toggle.

All three are implemented client-side — no new API endpoints or backend changes required.

---

## Prerequisites

* Projects page fetching data from `/api/v1/projects`
* `projects-filter.js` in place handling filtering and sorting
* Bootstrap 5 for dark mode class support

---

## Implementation Steps

1. Add search input to projects page and wire to the in-memory project list
2. Implement pagination — display N projects per page with prev/next controls
3. Add dark mode toggle — CSS variables, localStorage persistence, system preference detection

---

# 4.1 — Search

## How it works

The full project list is already fetched from the API and held in memory in `projects-filter.js`. Search filters that in-memory list on every keystroke — no network request needed.

## HTML — search input

```html
<input
  type="search"
  id="project-search"
  class="form-control"
  placeholder="Search projects..."
  aria-label="Search projects"
>
```

## JavaScript — search logic

In `projects-filter.js`, add a search filter to the existing pipeline:

```javascript
const searchInput = document.getElementById('project-search');

function filterProjects() {
  const query = searchInput.value.toLowerCase().trim();
  const category = activeCategory;
  const tag = activeTag;

  return allProjects.filter(project => {
    const matchesSearch = !query ||
      project.title.toLowerCase().includes(query) ||
      project.summary.toLowerCase().includes(query) ||
      project.tags.some(t => t.toLowerCase().includes(query));

    const matchesCategory = !category || project.category === category;
    const matchesTag = !tag || project.tags.includes(tag);

    return matchesSearch && matchesCategory && matchesTag;
  });
}

searchInput.addEventListener('input', () => renderProjects(filterProjects()));
```

Search matches against title, summary, and tags simultaneously.

---

# 4.2 — Pagination

## How it works

After filtering, only show N projects per page. Prev/Next buttons update the current page and re-render.

## JavaScript — pagination logic

```javascript
const PROJECTS_PER_PAGE = 12;
let currentPage = 1;

function paginate(projects) {
  const total = projects.length;
  const totalPages = Math.ceil(total / PROJECTS_PER_PAGE);
  const start = (currentPage - 1) * PROJECTS_PER_PAGE;
  const end = start + PROJECTS_PER_PAGE;

  renderCards(projects.slice(start, end));
  renderPaginationControls(currentPage, totalPages);
}

function renderPaginationControls(page, totalPages) {
  const container = document.getElementById('pagination');
  container.innerHTML = `
    <button class="btn btn-outline-secondary" ${page === 1 ? 'disabled' : ''}
      onclick="goToPage(${page - 1})">← Previous</button>
    <span class="mx-3">Page ${page} of ${totalPages}</span>
    <button class="btn btn-outline-secondary" ${page === totalPages ? 'disabled' : ''}
      onclick="goToPage(${page + 1})">Next →</button>
  `;
}

function goToPage(page) {
  currentPage = page;
  paginate(filterProjects());
  window.scrollTo({ top: 0, behavior: 'smooth' });
}
```

Reset to page 1 whenever the filter, category, tag, or search changes.

## HTML — pagination container

```html
<div id="pagination" class="d-flex justify-content-center align-items-center mt-4"></div>
```

---

# 4.3 — Dark Mode

## How it works

1. On page load, check `localStorage` for a saved preference
2. If none, check the system `prefers-color-scheme` media query
3. Apply the correct theme by adding/removing `data-theme="dark"` on `<html>`
4. A toggle button in the navbar switches and saves the preference

## CSS — colour variables for both themes

```css
:root {
  --color-bg: #ffffff;
  --color-surface: #f8f9fa;
  --color-text: #212529;
  --color-muted: #6c757d;
  --color-border: #dee2e6;
}

[data-theme="dark"] {
  --color-bg: #121212;
  --color-surface: #1e1e1e;
  --color-text: #e0e0e0;
  --color-muted: #9e9e9e;
  --color-border: #333333;
}

body {
  background-color: var(--color-bg);
  color: var(--color-text);
}
```

## JavaScript — theme toggle

```javascript
const THEME_KEY = 'theme';

function getPreferredTheme() {
  const saved = localStorage.getItem(THEME_KEY);
  if (saved) return saved;
  return window.matchMedia('(prefers-color-scheme: dark)').matches ? 'dark' : 'light';
}

function applyTheme(theme) {
  document.documentElement.setAttribute('data-theme', theme);
  localStorage.setItem(THEME_KEY, theme);
  updateToggleIcon(theme);
}

function toggleTheme() {
  const current = document.documentElement.getAttribute('data-theme');
  applyTheme(current === 'dark' ? 'light' : 'dark');
}

function updateToggleIcon(theme) {
  const btn = document.getElementById('theme-toggle');
  btn.textContent = theme === 'dark' ? '☀️' : '🌙';
}

// Apply on load
applyTheme(getPreferredTheme());
```

## HTML — toggle button in navbar

```html
<button id="theme-toggle" class="btn btn-sm btn-outline-secondary"
  onclick="toggleTheme()" aria-label="Toggle dark mode">
  🌙
</button>
```

Place the script that calls `applyTheme(getPreferredTheme())` in `<head>` (before `<body>` renders) to prevent flash of wrong theme on load.

---

# Files Involved

```bash
app/templates/projects.html          # Search input, pagination container
app/templates/base.html              # Dark mode toggle button, theme script in <head>
app/static/js/projects-filter.js     # Search and pagination logic
app/static/js/theme.js               # Dark mode toggle and persistence
app/static/css/main.css              # CSS variables for both themes
```

---

# Validation Checklist

* Search filters projects by title, summary, and tag in real time
* Search works in combination with category and tag filters
* Projects page shows 12 projects per page
* Prev/Next buttons navigate pages correctly
* Page resets to 1 when search or filters change
* Dark mode toggles correctly
* Dark mode preference persists on page reload
* Dark mode respects system `prefers-color-scheme` when no preference is saved
* No flash of wrong theme on page load

---

# Result

The projects page is now searchable, paginated, and supports dark mode. All three features work entirely client-side — no backend changes, no extra API calls. The projects page handles the full 51-project dataset efficiently and feels fast at every interaction.

---
